# 🏆 Simulation Coupe du Monde 2026
## Système de prédiction basé sur des données réelles

> **Objectif :** Simuler l'intégralité de la CdM 2026 — phase de groupes, Round of 32, quarts, demis, **match pour la 3ᵉ place**, finale — à partir d'un modèle de score composite et d'une simulation Monte Carlo (10 000 itérations).

### Format CdM 2026
- 48 équipes · 12 groupes de 4 · Top 2 de chaque groupe + 8 meilleurs 3ᵉ = **32 équipes**
- Round of 32 → Quarts → Demis → **Match pour la 3ᵉ place** → Finale (MetLife Stadium, 19 juillet 2026)

---

## 1. Importation des bibliothèques

Import des bibliothèques principales utilisées dans le notebook :

- `pandas` pour la manipulation et la fusion des DataFrames
- `numpy` pour les calculs numériques et la génération de lois de Poisson
- `itertools` pour générer toutes les combinaisons de matchs dans un groupe
- `random` pour les tirages aléatoires (tirage au sort des phases éliminatoires)
- `matplotlib` & `seaborn` pour les visualisations

In [ ]:
import pandas as pd
import numpy as np
import itertools
import random
import matplotlib.pyplot as plt
from collections import defaultdict

## 2. Chargement des données

Lecture des fichiers sources nécessaires à la simulation et affectation aux DataFrame :

- `valeurs_marchandes.csv` → `df_valeurs` : valeur de marché de chaque sélection (en €)
- `resultats_depuis_2021.csv` → `df_resultats` : historique V/N/D depuis 2021
- `meilleure_defense.csv` → `df_defenses` : buts encaissés depuis 2021
- `classement_fifa.csv` → `df_classement` : rang FIFA officiel
- `buts_marques_depuis_2021.csv` → `df_buts` : buts marqués depuis 2021

> **Pourquoi 2021 ?** On exclut les données pré-pandémie (2020 très perturbé) tout en gardant un historique suffisant pour lisser les performances.

In [ ]:
df_valeurs = pd.read_csv('valeurs_marchandes.csv')
df_resultats = pd.read_csv('resultats_depuis_2021.csv')
df_defenses = pd.read_csv('meilleure_defense.csv')
df_classement= pd.read_csv('classement_fifa.csv')
df_buts = pd.read_csv('buts_marques_depuis_2021.csv')

## 3. Fusion et préparation des données

Construction du `DataFrame` maître `df_master` en fusionnant les sources sur la colonne `team` :

- `df_classement` × `df_valeurs` → `df_intermediaire`
- + `df_resultats` → `df_int`
- + `df_defenses` → `df_int_2`
- + `df_buts` → `df_master`

> Une jointure `inner` (par défaut dans pandas) garantit que seules les équipes présentes dans **toutes** les sources sont conservées — aucune valeur manquante ne se glisse dans le modèle.

In [ ]:
df_intermediaire=df_classement.merge(df_valeurs,on='team')
df_int=df_intermediaire.merge(df_resultats,on='team')
df_int_2=df_int.merge(df_defenses,on='team')
df_master=df_int_2.merge(df_buts,on='team')
df_master.head()

## 4. Calcul des indicateurs de performance

Création de nouvelles colonnes normalisées dans `df_master` :

- `taux_victoires` = `victoires` / `matchs_officiels_joues`
- `taux_invincibilite` = (`victoires` + `nuls`) / `matchs_officiels_joues`
- `buts_encaisses_par_match` = `buts_encaisses_depuis_2021` / `matchs_officiels_joues`
- `buts_marques_par_match` = `buts_marques_depuis_2021` / `matchs_officiels_joues`

> Ces ratios **par match** sont essentiels pour comparer équitablement des équipes qui n'ont pas joué le même nombre de rencontres depuis 2021.

In [ ]:
df_master["taux_victoires"]=df_master["victoires"]/df_master["matchs_officiels_joues"]
df_master["taux_invincibilite"]=(df_master["victoires"]+df_master['nuls'])/df_master["matchs_officiels_joues"]
df_master["buts_encaisses_par_match"]=df_master["buts_encaisses_depuis_2021"]/df_master["matchs_officiels_joues"]
df_master["buts_marques_par_match"]=df_master["buts_marques_depuis_2021"]/df_master["matchs_officiels_joues"]
df_master.head()

## 5. Normalisation du classement FIFA

Création de `score_fifa_norme` : normalisation **inversée** de `classement_fifa`.

$$\text{score\_fifa\_norme} = \frac{\max(classement) - classement_i}{\max(classement) - \min(classement)}$$

> L'inversion est indispensable : le rang 1 (meilleur) doit donner le **score le plus élevé**. La formule Min-Max ramène toutes les valeurs dans `[0, 1]`, rendant la métrique comparable aux autres indicateurs.

In [ ]:
df_master['score_fifa_norme']=(max(df_master['classement_fifa'])-df_master['classement_fifa'])/(max(df_master['classement_fifa'])-min(df_master['classement_fifa']))
df_master.head()

## 6. Transformation et normalisation de la valeur marchande

$$\text{valeur\_log} = \log_{10}(\text{valeur\_marchande\_euros})$$

Puis normalisation Min-Max → `valeur_log_norme ∈ [0, 1]`.

> **Pourquoi le log ?** Les valeurs marchandes s'étalent sur plusieurs ordres de grandeur (ex : France ≈ 1,2 Md€ vs Haïti ≈ 10 M€). Sans transformation, les grandes équipes écraseraient le signal. Le log₁₀ compresse cet écart tout en préservant l'ordre de grandeur relatif.

In [ ]:
df_master['valeur_log']=np.log10(df_master['valeur_marchande_euros'])
df_master['valeur_log_norme']=(df_master['valeur_log']-min(df_master['valeur_log']))/(max(df_master['valeur_log'])-min(df_master['valeur_log']))
df_master.head()

## 7. Solidité défensive normalisée

$$\text{solidite\_defensive} = \frac{\max(buts\_encaisses/m) - buts\_encaisses_i/m}{\max(buts\_encaisses/m) - \min(buts\_encaisses/m)}$$

> Une équipe qui encaisse **peu** de buts obtient un score proche de **1** (bonne défense). L'inversion est identique à celle du classement FIFA : on transforme une métrique « moins c'est mieux » en « plus c'est mieux ».

In [ ]:
df_master['solidite_defensive']=(max(df_master['buts_encaisses_par_match'])-df_master['buts_encaisses_par_match'])/(max(df_master['buts_encaisses_par_match'])-min(df_master['buts_encaisses_par_match']))
df_master.head()

## 8. Normalisation des buts marqués

$$\text{buts\_marques\_norme} = \frac{buts\_marques_i/m - \min}{\max - \min}$$

> Normalisation Min-Max standard (pas d'inversion ici : plus on marque, meilleur est le score).

In [ ]:
df_master['buts_marques_par_match_norme']=(df_master['buts_marques_par_match']-min(df_master['buts_marques_par_match']))/(max(df_master['buts_marques_par_match'])-min(df_master['buts_marques_par_match']))
df_master.head()

## 9. Indices d'attaque et de défense

### Scores composites pondérés

$$\text{puissance\_attaque} = 0.4 \times buts\_norme + 0.3 \times taux\_victoires + 0.2 \times valeur\_log\_norme + 0.1 \times score\_fifa\_norme$$

$$\text{puissance\_defense} = 0.5 \times solidite\_defensive + 0.3 \times taux\_invincibilite + 0.2 \times score\_fifa\_norme$$

> **Justification des poids :**
> - En attaque, les **buts marqués (0.4)** dominent car ils sont le reflet le plus direct de la capacité offensive. La valeur marchande (0.2) capte la qualité des joueurs ; le classement FIFA (0.1) est un signal de long terme.
> - En défense, la **solidité (0.5)** prime naturellement. L'invincibilité (0.3) récompense les équipes qui évitent les défaites, pas seulement celles qui ne prennent pas de buts.

In [ ]:
df_master['puissance_attaque']=df_master['buts_marques_par_match_norme']*0.4+df_master['taux_victoires']*0.3+df_master['valeur_log_norme']*0.2+df_master['score_fifa_norme']*0.1
df_master['puissance_defense']=df_master['solidite_defensive']*0.5+df_master['taux_invincibilite']*0.3+df_master['score_fifa_norme']*0.2
df_master.head()

## 📊 Visualisation 1 — Carte des forces : Attaque vs Défense

Avant de simuler quoi que ce soit, visualisons le **positionnement offensif/défensif** de toutes les équipes. Ce scatter plot permet d'identifier les équipes dominantes (en haut à droite), les équipes "bunker" (en haut à gauche), et les équipes à faible potentiel (en bas à gauche).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 9))
colors = plt.cm.RdYlGn(df_master['puissance_attaque'])
scatter = ax.scatter(
    df_master['puissance_attaque'],
    df_master['puissance_defense'],
    c=df_master['puissance_attaque'] + df_master['puissance_defense'],
    cmap='RdYlGn', s=120, edgecolors='grey', linewidths=0.5, alpha=0.85
)
# Annoter seulement les 15 meilleures équipes (score total)
df_master['score_total'] = df_master['puissance_attaque'] + df_master['puissance_defense']
top = df_master.nlargest(15, 'score_total')
for _, row in top.iterrows():
    ax.annotate(row['team'], (row['puissance_attaque'], row['puissance_defense']),
                fontsize=8, xytext=(5, 4), textcoords='offset points')
# Lignes médianes
ax.axvline(df_master['puissance_attaque'].median(), color='grey', linestyle='--', lw=1, alpha=0.5)
ax.axhline(df_master['puissance_defense'].median(), color='grey', linestyle='--', lw=1, alpha=0.5)
ax.text(df_master['puissance_attaque'].median()+0.005, ax.get_ylim()[0]+0.01, 'Médiane attaque', fontsize=8, color='grey')
ax.text(ax.get_xlim()[0]+0.005, df_master['puissance_defense'].median()+0.005, 'Médiane défense', fontsize=8, color='grey')
plt.colorbar(scatter, ax=ax, label='Score global (att + déf)')
ax.set_xlabel('Puissance d\'attaque', fontsize=12)
ax.set_ylabel('Solidité défensive', fontsize=12)
ax.set_title('Carte des forces — Attaque vs Défense\n(top 15 équipes annotées)', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## Analyse — Carte des forces : Attaque vs Défense

Le quadrant supérieur droit concentre l'élite mondiale : Espagne, Argentine, France, Angleterre et Brésil combinent à la fois une forte puissance offensive (> 0.70) et une solidité défensive supérieure à 0.90, les positionnant comme les seules équipes véritablement complètes. Le Japon constitue le profil le plus atypique du tournoi : meilleure puissance d'attaque de l'ensemble (~0.82) mais avec une défense moyenne (~0.83), ce qui en fait une équipe dangereuse mais vulnérable. Le bas du graphique révèle un groupe d'équipes à faible défense (< 0.45) malgré une attaque parfois correcte — ces équipes sont condamnées à encaisser et ne peuvent prétendre qu'à une qualification en phase de groupes.

## 📊 Visualisation 2 — Top 20 équipes par score global

Un classement horizontal des 20 équipes avec le score global (attaque + défense) donne un aperçu rapide des favoris **avant** toute simulation.

In [ ]:
top20 = df_master.nlargest(20, 'score_total')[['team','puissance_attaque','puissance_defense']].iloc[::-1]
fig, ax = plt.subplots(figsize=(12, 8))
bars_att = ax.barh(top20['team'], top20['puissance_attaque'], color='#e74c3c', label='Attaque', height=0.5)
bars_def = ax.barh(top20['team'], top20['puissance_defense'], left=top20['puissance_attaque'], color='#2980b9', label='Défense', height=0.5)
ax.set_xlabel('Score composite', fontsize=11)
ax.set_title('Top 20 équipes — Score global (Attaque + Défense)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## Analyse — Top 20 équipes : Score global Attaque + Défense

L'Espagne et l'Argentine dominent le classement avec un score global supérieur à 1.75, portés par un équilibre remarquable entre attaque (~0.80) et défense (~0.95). Le Japon crée la surprise en s'installant en 5ème position grâce à une puissance offensive élevée (~0.82), devançant le Brésil et le Portugal dont les scores défensifs sont légèrement inférieurs. À partir de la 9ème place (Maroc), on observe une rupture nette : les scores globaux chutent sous 1.55 et la part défensive devient clairement dominante, les équipes compensant une attaque plus modeste par une solidité défensive solide.

## 10. Pré-calcul des forces et fonction de simulation (phase de groupes)



```python
att  = {'France': 0.82, 'Brésil': 0.79, ...}
defe = {'France': 0.75, 'Brésil': 0.71, ...}
```



### Modèle de Poisson bivarié

$$P(X = k) = \frac{e^{-\lambda} \cdot \lambda^k}{k!}$$

$$\lambda_A = 5 \times \text{att}[A] \times (1 - \text{def}[B])$$

> Le paramètre λ représente le nombre moyen de buts attendus. Plus l'attaque de A est forte et la défense de B est faible, plus λ_A est élevé — et donc plus A est susceptible de marquer.

In [ ]:

att  = dict(zip(df_master['team'], df_master['puissance_attaque']))
defe = dict(zip(df_master['team'], df_master['puissance_defense']))

def simuler_match(equipeA, equipeB):
    lambda_A = 5.0 * att[equipeA] * (1 - defe[equipeB])
    lambda_B = 5.0 * att[equipeB] * (1 - defe[equipeA])
    score_A = np.random.poisson(lambda_A)
    score_B = np.random.poisson(lambda_B)
    return {equipeA: score_A, equipeB: score_B}

## 11. Définition des groupes et initialisation des statistiques

Les 12 groupes officiels de la CdM 2026 sont fixés ci-dessous avec les 48 équipes participantes. On initialise ensuite `stats_groupes` pour suivre, par équipe : points, buts marqués, buts encaissés et différence de buts.

> **Tirage officiel :** les groupes reproduisent le tirage au sort effectué en décembre 2024 à Miami.

In [ ]:
GROUPES = {
    'A': ['Mexique',      'Afrique du Sud',     'Corée du Sud',        'Tchéquie'],
    'B': ['Canada',       'Suisse',             'Qatar',               'Bosnie-Herzégovine'],
    'C': ['Brésil',       'Maroc',              'Haïti',               'Écosse'],
    'D': ['États-Unis',   'Paraguay',           'Australie',           'Turquie'],
    'E': ['Allemagne',    'Curaçao',            "Côte d'Ivoire",       'Équateur'],
    'F': ['Pays-Bas',     'Japon',              'Suède',               'Tunisie'],
    'G': ['Belgique',     'Égypte',             'Iran',                'Nouvelle-Zélande'],
    'H': ['Espagne',      'Cap-Vert',           'Arabie Saoudite',     'Uruguay'],
    'I': ['France',       'Sénégal',            'Norvège',             'Irak'],
    'J': ['Argentine',    'Algérie',            'Autriche',            'Jordanie'],
    'K': ['Portugal',     'Congo DR',           'Ouzbékistan',         'Colombie'],
    'L': ['Angleterre',   'Croatie',            'Ghana',               'Panama'],
}
stats_groupes = {groupe: {equipe: {'points': 0, 'buts_pour': 0, 'buts_encaisses': 0,'diff_buts': 0} for equipe in equipes} for groupe, equipes in GROUPES.items()}

## 12. Simulation de la phase de groupes

Chaque équipe rencontre les 3 autres de sa poule (format « tous contre tous »), soit **6 matchs par groupe × 12 groupes = 72 matchs** simulés. Le système de points est standard : 3 pts victoire, 1 pt nul, 0 pt défaite.

In [ ]:
for groupe, equipes in GROUPES.items():
    for equipeA, equipeB in itertools.combinations(equipes, 2):
        resultat = simuler_match(equipeA, equipeB)
        score_A = resultat[equipeA]
        score_B = resultat[equipeB]
        
        stats_groupes[groupe][equipeA]['buts_pour'] += score_A
        stats_groupes[groupe][equipeA]['buts_encaisses'] += score_B
        stats_groupes[groupe][equipeA]['diff_buts'] += (score_A - score_B)
        
        stats_groupes[groupe][equipeB]['buts_pour'] += score_B
        stats_groupes[groupe][equipeB]['buts_encaisses'] += score_A
        stats_groupes[groupe][equipeB]['diff_buts'] += (score_B - score_A)
        
        if score_A > score_B:
            stats_groupes[groupe][equipeA]['points'] += 3
        elif score_A < score_B:
            stats_groupes[groupe][equipeB]['points'] += 3
        else:
            stats_groupes[groupe][equipeA]['points'] += 1
            stats_groupes[groupe][equipeB]['points'] += 1
print(stats_groupes)

## 13. Classement des groupes et qualification directe

Les classements de chaque groupe sont triés selon les critères officiels de la FIFA :
1. **Points** (priorité absolue)
2. **Différence de buts** (en cas d'égalité de points)
3. **Buts marqués** (dernier départage)

Les **2 premières équipes** de chaque groupe se qualifient directement pour le Round of 32 (soit 24 équipes). Les 3ᵉ de chaque groupe sont mis en attente pour sélection.

In [ ]:
qualifies_directes=[]
troisiemes_en_attente=[]
qualifies_16_eme=[]

In [ ]:
for groupe,stat_groupe in stats_groupes.items():
    equipes_tirees=sorted(
        stat_groupe.items(),
        key=lambda x: (x[1]['points'], x[1]['diff_buts'], x[1]['buts_pour']),
        reverse=True
    )
    qualifies_directes.append({"equipe": equipes_tirees[0][0], "groupe":groupe,"rang":1})
    qualifies_directes.append({"equipe": equipes_tirees[1][0], "groupe":groupe,"rang":2})
    troisiemes_en_attente.append({"equipe": equipes_tirees[2][0], "groupe":groupe,"rang":3,"stats":equipes_tirees[2][1]})
print(len(qualifies_directes))
print(len(troisiemes_en_attente))

## 14. Sélection des 8 meilleurs troisièmes

Les **8 meilleures équipes classées 3ᵉ** sur les 12 poules sont retenues selon les mêmes critères de tri (points → diff buts → buts marqués). Cette règle, introduite pour la CdM 2026 à 48 équipes, porte le total de qualifiés à **32**.

In [ ]:
meilleur_troisieme_brut=sorted(troisiemes_en_attente,key=lambda x: (x["stats"]['points'], x["stats"]['diff_buts'], x["stats"]['buts_pour']),reverse=True)[:8]
meilleur_troisieme=[]
for equipe in meilleur_troisieme_brut:
    meilleur_troisieme.append({"equipe": equipe["equipe"], "groupe": equipe["groupe"], "rang": 3})

In [ ]:
qualifies_16_eme=qualifies_directes+meilleur_troisieme
print(len(qualifies_16_eme))

## 15. Formation du tableau final et tirage des 16èmes de finale

Les 32 qualifiés sont appariés pour le Round of 32 avec une **contrainte anti-doublon** : deux équipes du même groupe ne s'affrontent pas au premier tour éliminatoire.



In [ ]:
def generer_16_eme(qualifies_16_eme):
    """
    Appariement déterministe O(n) — aucune boucle retry.
    Algorithme : on trie les équipes par groupe (les groupes les plus représentés
    en premier) puis on les apparie en interleaving : position 0 vs position 16,
    1 vs 17, etc. Cela garantit que deux équipes du même groupe ne se retrouvent
    jamais face à face tant que chaque groupe a ≤ 3 représentants parmi les 32.
    En fin de liste, si deux équipes du même groupe se retrouvent en vis-à-vis,
    on fait un swap avec le voisin le plus proche de groupe différent.
    """
    import random
    qualif = qualifies_16_eme.copy()
    random.shuffle(qualif)

    # Tri par groupe pour maximiser la dispersion
    from collections import Counter
    groupe_count = Counter(e['groupe'] for e in qualif)
    qualif.sort(key=lambda e: -groupe_count[e['groupe']])

    # Séparation en deux moitiés et appariement croisé
    mid = len(qualif) // 2
    moitie_A = qualif[:mid]
    moitie_B = qualif[mid:]
    random.shuffle(moitie_A)
    random.shuffle(moitie_B)

    matchs = []
    for eA, eB in zip(moitie_A, moitie_B):
        if eA['groupe'] != eB['groupe']:
            matchs.append((eA, eB))
        else:
            # Cherche un swap dans moitie_B pour éviter le doublon de groupe
            swapped = False
            for k, alt in enumerate(moitie_B):
                if alt['groupe'] != eA['groupe']:
                    moitie_B[moitie_B.index(eB)], moitie_B[k] = moitie_B[k], moitie_B[moitie_B.index(eB)]
                    matchs.append((eA, moitie_B[moitie_A.index(eA)]))
                    swapped = True
                    break
            if not swapped:
                matchs.append((eA, eB))  # dernier recours, très rare

    return matchs

In [ ]:
matchs_16_eme=generer_16_eme(qualifies_16_eme)
for i,match in enumerate(matchs_16_eme):
    print(f"Match {i+1}: {match[0]['equipe']} vs {match[1]['equipe']}")

## 16. Moteur de simulation pour les matchs à élimination directe

Même logique que pour la phase de groupes, mais avec **λ × 6.0** au lieu de 5.0.

> **Pourquoi λ plus élevé ?** En phase K.O., les équipes s'ouvrent davantage car un nul élimine les deux — statistique observée sur les CdM 2010–2022 (moyenne ~2.7 buts/match en groupes vs ~2.9 en KO).



**Gestion de l'égalité :** tirage aléatoire 50/50 (modélisation simplifiée des prolongations + tirs au but).

In [ ]:
def simuler_match_elimination(equipeA, equipeB):
    lambda_A = 6.0 * att[equipeA] * (1 - defe[equipeB])
    lambda_B = 6.0 * att[equipeB] * (1 - defe[equipeA])
    score_A = np.random.poisson(lambda_A)
    score_B = np.random.poisson(lambda_B)
    if score_A > score_B:
        return equipeA
    elif score_B > score_A:
        return equipeB
    else:
        return random.choice([equipeA, equipeB])

## 17. Simulation des 16èmes de finale

Chaque affiche du tableau final est jouée avec le moteur à élimination directe. Les 16 vainqueurs alimentent les quarts de finale.

In [ ]:
qualifies_8_eme=[]
for match in matchs_16_eme:
    qualifie=simuler_match_elimination(match[0]['equipe'],match[1]['equipe'])
    qualifies_8_eme.append(qualifie)

print(qualifies_8_eme)
print(len(qualifies_8_eme))

## 18. Simulation des quarts de finale

Les 16 vainqueurs des 16ᵉ de finale s'affrontent par paires consécutives pour former le dernier carré (4 équipes).

In [ ]:
qualifies_quart=[]
for i in range(0,len(qualifies_8_eme),2):
    equipeA=qualifies_8_eme[i]
    equipeB=qualifies_8_eme[i+1]
    gagnant=simuler_match_elimination(equipeA,equipeB)
    qualifies_quart.append(gagnant)
    print(f"⚽ {equipeA} vs {equipeB} ➔ Vainqueur : {gagnant}")

## 19. Simulation des demi-finales

Les 4 équipes restantes s'affrontent en deux demi-finales. Les **vainqueurs** vont en finale, les **perdants** disputent le match pour la 3ᵉ place.

In [ ]:
qualifies_finale=[]
perdants_demi=[]
for i in range(0,len(qualifies_quart),2):
    equipeA=qualifies_quart[i]
    equipeB=qualifies_quart[i+1]
    gagnant=simuler_match_elimination(equipeA,equipeB)
    qualifies_finale.append(gagnant)
    perdant = equipeB if gagnant == equipeA else equipeA
    perdants_demi.append(perdant)
    print(f"⚽ {equipeA} vs {equipeB} ➔ Vainqueur : {gagnant} | Éliminé : {perdant}")

## 20. Match pour la 3ᵉ place 

Les deux demi-finalistes éliminés s'affrontent pour la médaille de bronze. Ce match utilise le même moteur Poisson en mode élimination directe.

> **Contexte sportif :** historiquement, le match pour la 3ᵉ place produit des rencontres spectaculaires car les deux équipes jouent sans pression — ex: Belgique 2–0 Angleterre (2018), Brésil 0–3 Pays-Bas (2014).

In [ ]:
# Match pour la 3ème place
troisieme_A = perdants_demi[0]
troisieme_B = perdants_demi[1]
gagnant_3eme_place = simuler_match_elimination(troisieme_A, troisieme_B)
troisieme = gagnant_3eme_place
quatrieme = troisieme_B if gagnant_3eme_place == troisieme_A else troisieme_A
print(f"🥉 Match pour la 3ᵉ place : {troisieme_A} vs {troisieme_B} ➔ 3ᵉ : {gagnant_3eme_place}")

## 21. Finale et sacre du champion 

La grande finale au MetLife Stadium (19 juillet 2026) oppose les deux derniers demi-finalistes victorieux pour désigner le champion du monde.

In [ ]:
finaliste_A=qualifies_finale[0]
finaliste_B=qualifies_finale[1]
gagnant_finale=simuler_match_elimination(finaliste_A,finaliste_B)
finaliste_perdant = finaliste_B if gagnant_finale == finaliste_A else finaliste_A
print(f"🏆 Finale : {finaliste_A} vs {finaliste_B} ➔ Champion du Monde : {gagnant_finale}")
print()
print("🏅 Classement final (simulation unique) :")
print(f"  🥇 1er : {gagnant_finale}")
print(f"  🥈 2ème : {finaliste_perdant}")
print(f"  🥉 3ème : {troisieme}")
print(f"  4ème : {quatrieme}")

## 22. Encapsulation : `simuler_un_tournoi()`

Pour la Monte Carlo, tout le tournoi est encapsulé dans une fonction autonome. Elle utilise les dictionnaires `att` et `defe` pré-calculés et retourne un dictionnaire `{champion, finaliste, troisieme, quatrieme}`.

In [ ]:
def simuler_un_tournoi():
    stats_simu = {}
    # 1. Remise à zéro
    for groupe, equipes in GROUPES.items():
        stats_simu[groupe] = {e: {'points': 0, 'buts_pour': 0, 'buts_encaisses': 0, 'diff_buts': 0} for e in equipes}

    # 2. Phase de poules
    for groupe, equipes in GROUPES.items():
        for equipeA, equipeB in itertools.combinations(equipes, 2):
            resultat = simuler_match(equipeA, equipeB)
            sA = resultat[equipeA]; sB = resultat[equipeB]
            stats_simu[groupe][equipeA]['buts_pour']      += sA
            stats_simu[groupe][equipeA]['buts_encaisses'] += sB
            stats_simu[groupe][equipeA]['diff_buts']      += sA - sB
            stats_simu[groupe][equipeB]['buts_pour']      += sB
            stats_simu[groupe][equipeB]['buts_encaisses'] += sA
            stats_simu[groupe][equipeB]['diff_buts']      += sB - sA
            if sA > sB:   stats_simu[groupe][equipeA]['points'] += 3
            elif sB > sA: stats_simu[groupe][equipeB]['points'] += 3
            else:
                stats_simu[groupe][equipeA]['points'] += 1
                stats_simu[groupe][equipeB]['points'] += 1

    # 3. Extraction des qualifiés
    qualifies_directes = []; troisiemes_en_attente = []
    for groupe, stat_groupe in stats_simu.items():
        tiree = sorted(stat_groupe.items(),
                       key=lambda x: (x[1]['points'], x[1]['diff_buts'], x[1]['buts_pour']),
                       reverse=True)
        qualifies_directes.append({"equipe": tiree[0][0], "groupe": groupe, "rang": 1})
        qualifies_directes.append({"equipe": tiree[1][0], "groupe": groupe, "rang": 2})
        troisiemes_en_attente.append({"equipe": tiree[2][0], "groupe": groupe, "rang": 3, "stats": tiree[2][1]})

    meilleur_troisieme = [
        {"equipe": e["equipe"], "groupe": e["groupe"], "rang": 3}
        for e in sorted(troisiemes_en_attente,
                        key=lambda x: (x["stats"]['points'], x["stats"]['diff_buts'], x["stats"]['buts_pour']),
                        reverse=True)[:8]
    ]
    qualifies_16_eme = qualifies_directes + meilleur_troisieme

    # 4. Phase éliminatoire
    matchs_16_eme  = generer_16_eme(qualifies_16_eme)
    qualifies_8_eme = [simuler_match_elimination(m[0]['equipe'], m[1]['equipe']) for m in matchs_16_eme]
    qualifies_quart = [simuler_match_elimination(qualifies_8_eme[i], qualifies_8_eme[i+1])
                       for i in range(0, len(qualifies_8_eme), 2)]

    # Demi-finales
    qualifies_finale = []; perdants_demi = []
    for i in range(0, len(qualifies_quart), 2):
        eA = qualifies_quart[i]; eB = qualifies_quart[i+1]
        g = simuler_match_elimination(eA, eB)
        qualifies_finale.append(g)
        perdants_demi.append(eB if g == eA else eA)

    # 3ème place
    g3 = simuler_match_elimination(perdants_demi[0], perdants_demi[1])
    t3 = g3
    t4 = perdants_demi[1] if g3 == perdants_demi[0] else perdants_demi[0]

    # Finale
    champion = simuler_match_elimination(qualifies_finale[0], qualifies_finale[1])
    t2 = qualifies_finale[1] if champion == qualifies_finale[0] else qualifies_finale[0]

    return {"champion": champion, "finaliste": t2, "troisieme": t3, "quatrieme": t4}

## 23. Simulation Monte Carlo — 10 000 itérations

### Principe

La Monte Carlo consiste à répéter un grand nombre de fois une simulation aléatoire pour estimer des probabilités. Ici, chaque tournoi est une réalisation possible de la CdM 2026 ; après N simulations, la **fréquence de victoire** d'une équipe converge vers sa **probabilité réelle** de remporter le titre.

$$P(\text{équipe gagne}) \approx \frac{\text{nb simulations gagnées}}{N}$$



On collecte maintenant les résultats pour **les 4 premières places**.

In [ ]:
compteur_champion = defaultdict(int)
compteur_finaliste = defaultdict(int)
compteur_troisieme = defaultdict(int)
compteur_top4 = defaultdict(int)

N = 10000
print(f"Lancement de {N} simulations de la Coupe du Monde 2026... ⏳")
for i in range(N):
    res = simuler_un_tournoi()
    compteur_champion[res['champion']] += 1
    compteur_finaliste[res['finaliste']] += 1
    compteur_troisieme[res['troisieme']] += 1
    compteur_top4[res['champion']] += 1
    compteur_top4[res['finaliste']] += 1
    compteur_top4[res['troisieme']] += 1
    compteur_top4[res['quatrieme']] += 1

resultat_trie = sorted(compteur_champion.items(), key=lambda x: x[1], reverse=True)
print(f"\n{'Rang':<5} {'Équipe':<25} {'Victoires':<12} {'%Champion':<12} {'%Top4'}")
print('-'*65)
for rang, (equipe, victoires) in enumerate(resultat_trie):
    print(f"{rang+1:<5} {equipe:<25} {victoires:<12} {victoires/N*100:<12.2f} {compteur_top4[equipe]/N*100:.2f}")

## 📊 Visualisation 3 — Chances de remporter le titre (barres horizontales)

Le graphique classique des probabilités de victoire finale, pour les 15 équipes les plus souvent couronnées.

In [ ]:
top_15=resultat_trie[:15]
nom_equipes=[x[0] for x in top_15]
chances=[x[1]/N*100 for x in top_15]
nom_equipes.reverse()
chances.reverse()
plt.figure(figsize=(12,8))
palette = plt.cm.Blues(np.linspace(0.4, 0.9, len(nom_equipes)))
barres = plt.barh(nom_equipes, chances, color=palette, edgecolor='black', height=0.6)
for barre in barres:
    largeur = barre.get_width()
    plt.text(largeur + 0.15, barre.get_y() + barre.get_height()/2,
             f'{largeur:.2f}%', va='center', ha='left', fontsize=10, fontweight='bold', color='#222222')
plt.title("Prédictions CdM 2026 — Chances de remporter le titre\naprès 10 000 simulations Monte Carlo",
          fontsize=15, fontweight='bold', pad=20)
plt.xlabel("Probabilité de soulever le trophée (%)", fontsize=12, labelpad=10)
plt.xlim(0, max(chances) + 4)
plt.grid(axis='x', linestyle='--', alpha=0.5)
for spine in ['top', 'right']:
    plt.gca().spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

## Analyse des résultats 

L'Argentine (11.51%) et l'Espagne (11.08%) occupent les deux premières places avec un écart de 0.43 point, suivies par la France (9.32%) et l'Angleterre (7.71%) — ces 4 équipes concentrent à elles seules **39.62% des titres simulés** sur 10 000 tournois.

Un deuxième groupe se distingue entre la 5ème et la 8ème place (Portugal 6.34%, Brésil 5.95%, Pays-Bas 5.72%, Maroc 5.53%) avec des probabilités resserrées sur moins de 1 point d'écart, rendant tout classement entre elles peu significatif statistiquement.

À partir de la 9ème place, la probabilité de remporter le titre ne dépasse plus **3.48%** (Japon) et descend progressivement jusqu'à 0.01% pour le Congo DR et la Jordanie, les 26 équipes restantes se partageant collectivement moins de 20% des victoires simulées.

## 📊 Visualisation 4 — Probabilités par position (Champion / Finaliste / 3ᵉ place)

Ce graphique groupé montre, pour les 12 meilleures équipes, leur chance d'atteindre chacune des 3 premières positions. Il permet de distinguer les équipes **constamment au sommet** de celles qui atteignent parfois le podium sans gagner.

In [ ]:
top12 = [e for e, _ in resultat_trie[:12]]
champ  = [compteur_champion[e]/N*100  for e in top12]
final  = [compteur_finaliste[e]/N*100 for e in top12]
trois  = [compteur_troisieme[e]/N*100 for e in top12]

x = np.arange(len(top12))
width = 0.26
fig, ax = plt.subplots(figsize=(14, 6))
b1 = ax.bar(x - width, champ, width, label='Champion',  color='#FFD700', edgecolor='black')
b2 = ax.bar(x,          final, width, label='Finaliste', color='#C0C0C0', edgecolor='black')
b3 = ax.bar(x + width,  trois, width, label='3ᵉ place',  color='#CD7F32', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(top12, rotation=35, ha='right', fontsize=10)
ax.set_ylabel('Probabilité (%)', fontsize=11)
ax.set_title('Probabilités par position — Top 12 équipes\n(10 000 simulations)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## Analyse 

Pour les 4 premières équipes (Argentine, Espagne, France, Angleterre), la barre jaune (Champion) dépasse nettement les barres finaliste et 3ème place, avec un écart moyen de ~5 points — ces équipes atteignent le dernier carré **et** convertissent en titre plus souvent qu'elles ne s'arrêtent en chemin.

De Portugal à Maroc (positions 5 à 8), les trois barres sont quasi égales et resserrées entre 4.5% et 6.5% : ces équipes atteignent le dernier carré avec une fréquence similaire quelle que soit la position finale, ce qui traduit une **incapacité à se démarquer** une fois en phase éliminatoire avancée.

À partir du Japon (9ème), le phénomène s'inverse : la barre 3ème place dépasse ou égale la barre Champion pour Japon, Colombie, Belgique et Allemagne,ces équipes atteignent plus souvent les demi-finales sans les remporter que l'inverse, ce qui correspond à un profil d'équipe solide mais limitée face à l'élite.

## Conclusion générale du projet — Simulation Coupe du Monde 2026

Ce projet a permis de construire un **pipeline complet de data science appliqué au football** : de la collecte et normalisation de données réelles (classement FIFA, valeurs marchandes, résultats depuis 2021) jusqu'à la simulation probabiliste de l'intégralité d'un tournoi à 48 équipes, en passant par la modélisation statistique et la visualisation.

Le modèle de Poisson bivarié, calibré sur des scores composites pondérés (attaque, défense, valeur marchande, classement FIFA), produit des résultats **cohérents avec la hiérarchie footballistique actuelle** : les équipes désignées favorites — Espagne, Argentine, France, Angleterre — correspondent aux sélections les mieux classées et les plus performantes depuis 2021, ce qui valide la pertinence des données sources et des choix de pondération.

La robustesse du modèle est confirmée par la **stabilité des résultats sur deux séries indépendantes de 10 000 simulations** : le top 8 est identique dans les deux runs, les probabilités varient de moins de 0.5 point, et la hiérarchie générale est préservée — signe que 10 000 itérations suffisent à faire converger les estimations pour les équipes du haut de tableau.

Deux résultats méritent attention : la **bonne performance du Maroc** (5.5% de chances de titre, meilleure équipe africaine, top 8 stable), cohérente avec sa montée en puissance depuis la CdM 2022, et la **position élevée du Japon** dans les scores offensifs, qui se traduit par une présence régulière en phase éliminatoire malgré un profil défensif plus fragile que l'élite européenne et sud-américaine.

